# L11: 工具参数解析与验证

> 深入理解如何从 LLM 的自然语言中提取结构化参数

In [1]:
# === 环境设置 ===
import sys
import os
import json
import asyncio
import re
from pathlib import Path

# 自动找到项目根目录
current_path = Path(os.getcwd())
project_path = None

for path in [current_path] + list(current_path.parents):
    if (path / "backend").exists():
        project_path = str(path)
        break

if project_path and project_path not in sys.path:
    sys.path.insert(0, project_path)

# 加载 .env
env_file = Path(project_path) / ".env" if project_path else None
if env_file and env_file.exists():
    with open(env_file, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line and not line.startswith("#") and "=" in line:
                key, value = line.split("=", 1)
                os.environ[key.strip()] = value.strip()

print(f"项目路径: {project_path}")
print(f"Python 版本: {sys.version.split()[0]}")

from backend.tools.base import BaseTool
from backend.tools.registry import ToolRegistry, register_tool
from backend.llm.models import Message, ToolCall, FunctionCall
print("✓ 模块导入成功")

项目路径: c:\Users\zying\Desktop\pro_agent\agent-learning-project
Python 版本: 3.14.4
✓ 模块导入成功


## 11.1 参数解析的挑战

### 从自然语言到结构化参数

```
用户: "帮我查一下北京明天的天气"
             ↓
LLM 理解 → 需要调用 get_weather 工具
             ↓
提取参数 → city="北京", date="明天"
             ↓
参数转换 → date="明天" → "2024-01-15"
```

### 参数解析的三种方式

| 方式 | LLM 输出 | 解析难度 | 可靠性 |
|------|---------|---------|--------|
| **Native Function Calling** | 结构化 JSON | 低 | 高 |
| **Prompt 引导** | 文本 + JSON | 中 | 中 |
| **自由文本** | 纯自然语言 | 高 | 低 |

## 11.2 JSON Schema 参数定义

### 标准参数定义格式

```python
parameters = {
    "param_name": {
        "type": "string|number|boolean|array|object",
        "description": "参数描述（LLM 会看到）",
        "required": True|False,
        "default": "默认值",
        "enum": ["可选值1", "可选值2"],  # 可选：枚举限制
        "min": 0,     # 可选：最小值
        "max": 100    # 可选：最大值
    }
}`
```

### 类型映射

| JSON Schema 类型 | Python 类型 | 示例 |
|-----------------|-------------|------|
| string | str | "hello" |
| number | int/float | 42, 3.14 |
| boolean | bool | true, false |
| array | list | [1, 2, 3] |
| object | dict | {"key": "value"} |

## 11.3 查看项目中的参数定义

In [2]:
# 查看现有工具的参数定义
from backend.tools.search import DuckDuckGoSearchTool, WikipediaSearchTool
from backend.tools.calculator import CalculatorTool

# 创建工具实例
ddg_tool = DuckDuckGoSearchTool()
wiki_tool = WikipediaSearchTool()
calc_tool = CalculatorTool()

print("=== DuckDuckGoSearchTool 参数定义 (JSON Schema) ===\n")
print(json.dumps(ddg_tool.get_schema(), indent=2, ensure_ascii=False))

print("\n=== WikipediaSearchTool 参数定义 (JSON Schema) ===\n")
print(json.dumps(wiki_tool.get_schema(), indent=2, ensure_ascii=False))

print("\n=== CalculatorTool 参数定义 (JSON Schema) ===\n")
print(json.dumps(calc_tool.get_schema(), indent=2, ensure_ascii=False))

print("\n\n=== 原始 ToolParameter 列表格式 ===\n")
for param in calc_tool.parameters:
    print(f"{param.name}: {param.type} (required={param.required})")

=== DuckDuckGoSearchTool 参数定义 (JSON Schema) ===

{
  "type": "object",
  "properties": {
    "query": {
      "type": "string",
      "description": "搜索查询内容"
    },
    "num_results": {
      "type": "number",
      "description": "返回结果数量，默认5",
      "default": 5
    }
  },
  "required": [
    "query"
  ]
}

=== WikipediaSearchTool 参数定义 (JSON Schema) ===

{
  "type": "object",
  "properties": {
    "query": {
      "type": "string",
      "description": "搜索查询内容"
    },
    "sentences": {
      "type": "number",
      "description": "返回结果的句子数量，默认3",
      "default": 3
    }
  },
  "required": [
    "query"
  ]
}

=== CalculatorTool 参数定义 (JSON Schema) ===

{
  "type": "object",
  "properties": {
    "expression": {
      "type": "string",
      "description": "要计算的数学表达式，例如: 2 + 2, 10 * (5 + 3), 100 / 4"
    }
  },
  "required": [
    "expression"
  ]
}


=== 原始 ToolParameter 列表格式 ===

expression: string (required=True)


## 11.4 参数验证

### 验证层次

```
第 1 层: 类型检查 (type 必须匹配)
   ↓
第 2 层: 必填检查 (required 参数不能缺失)
   ↓
第 3 层: 值域检查 (min/max/enum 约束)
   ↓
第 4 层: 业务逻辑验证 (execute 内部)
```

### 实现参数验证器

In [3]:
from typing import Any, Dict, List, Optional

class ParameterValidator:
    """参数验证器 - 支持 JSON Schema 格式"""
    
    @staticmethod
    def validate(params: Dict[str, Any], schema: Dict[str, Any], required: List[str] = None) -> tuple[bool, Optional[str]]:
        """
        验证参数是否符合 JSON Schema
        
        Args:
            params: 实际参数
            schema: 参数定义 (properties 字典)
            required: 必填参数名列表 (JSON Schema 格式)
            
        Returns:
            (是否有效, 错误信息)
        """
        required = required or []
        
        # 1. 检查必填参数 (JSON Schema: required 是列表)
        for param_name in required:
            if param_name not in params:
                return False, f"缺少必填参数: {param_name}"
        
        # 2. 检查类型和值域
        for param_name, param_value in params.items():
            if param_name not in schema:
                continue
            
            param_def = schema[param_name]
            
            # 类型检查
            expected_type = param_def.get("type")
            if not ParameterValidator._check_type(param_value, expected_type):
                return False, f"参数 {param_name} 类型错误: 期望 {expected_type}, 实际 {type(param_value).__name__}"
            
            # 枚举检查
            if "enum" in param_def and param_value not in param_def["enum"]:
                return False, f"参数 {param_name} 值无效: {param_value}, 允许的值: {param_def['enum']}"
            
            # 数值范围检查 (JSON Schema 使用 minimum/maximum)
            if isinstance(param_value, (int, float)):
                if "minimum" in param_def and param_value < param_def["minimum"]:
                    return False, f"参数 {param_name} 小于最小值 {param_def['minimum']}"
                if "maximum" in param_def and param_value > param_def["maximum"]:
                    return False, f"参数 {param_name} 超过最大值 {param_def['maximum']}"
        
        return True, None
    
    @staticmethod
    def _check_type(value: Any, expected_type: str) -> bool:
        """检查值是否符合预期类型"""
        type_map = {
            "string": str,
            "number": (int, float),
            "integer": int,
            "boolean": bool,
            "array": list,
            "object": dict
        }
        expected_python_type = type_map.get(expected_type)
        if expected_python_type is None:
            return True  # 未知类型，跳过检查
        return isinstance(value, expected_python_type)

# 测试验证器 (使用 JSON Schema 格式)
test_properties = {
    "query": {
        "type": "string",
        "description": "搜索查询"
    },
    "limit": {
        "type": "number",
        "description": "结果数量",
        "minimum": 1,
        "maximum": 100
    },
    "category": {
        "type": "string",
        "description": "分类",
        "enum": ["news", "blog", "video"]
    }
}
test_required = ["query"]

# 测试用例
test_cases = [
    ({"query": "test"}, True, "正常情况"),
    ({}, False, "缺少必填参数"),
    ({"query": "test", "limit": 0}, False, "低于最小值"),
    ({"query": "test", "limit": 101}, False, "超过最大值"),
    ({"query": "test", "category": "other"}, False, "枚举值无效"),
    ({"query": "test", "limit": 10}, True, "所有验证通过"),
]

print("=== 参数验证测试 (JSON Schema 格式) ===\n")
for params, expected_valid, description in test_cases:
    is_valid, error = ParameterValidator.validate(params, test_properties, test_required)
    status = "✓" if is_valid == expected_valid else "✗"
    print(f"{status} {description}")
    if not is_valid:
        print(f"   错误: {error}")

=== 参数验证测试 (JSON Schema 格式) ===

✓ 正常情况
✓ 缺少必填参数
   错误: 缺少必填参数: query
✓ 低于最小值
   错误: 参数 limit 小于最小值 1
✓ 超过最大值
   错误: 参数 limit 超过最大值 100
✓ 枚举值无效
   错误: 参数 category 值无效: other, 允许的值: ['news', 'blog', 'video']
✓ 所有验证通过


## 11.5 参数类型转换

### 为什么需要类型转换？

LLM 可能输出字符串类型的数字：`"123"`，但工具需要 `int` 类型。

In [4]:
class ParameterConverter:
    """参数类型转换器"""
    
    @staticmethod
    def convert(params: Dict[str, Any], schema: Dict[str, Any]) -> Dict[str, Any]:
        """
        根据类型定义转换参数
        """
        converted = params.copy()
        
        for param_name, param_value in params.items():
            if param_name not in schema:
                continue
            
            param_def = schema[param_name]
            expected_type = param_def.get("type")
            
            # 如果类型已经匹配，不需要转换
            if ParameterValidator._check_type(param_value, expected_type):
                continue
            
            # 尝试转换
            try:
                if expected_type == "number" or expected_type == "integer":
                    if isinstance(param_value, str):
                        converted[param_name] = float(param_value) if "." in param_value else int(param_value)
                elif expected_type == "boolean":
                    if isinstance(param_value, str):
                        converted[param_name] = param_value.lower() in ("true", "1", "yes")
            except (ValueError, TypeError):
                # 转换失败，保持原值
                pass
        
        return converted

# 测试转换器
print("=== 参数类型转换测试 ===\n")

test_params = {
    "count": "42",       # 字符串数字
    "ratio": "3.14",     # 字符串浮点数
    "enabled": "true",   # 字符串布尔值
    "name": "test"       # 已经是字符串
}

test_schema = {
    "count": {"type": "integer"},
    "ratio": {"type": "number"},
    "enabled": {"type": "boolean"},
    "name": {"type": "string"}
}

converted = ParameterConverter.convert(test_params, test_schema)

print("转换前:")
for k, v in test_params.items():
    print(f"  {k}: {v} ({type(v).__name__})")

print("\n转换后:")
for k, v in converted.items():
    print(f"  {k}: {v} ({type(v).__name__})")

=== 参数类型转换测试 ===

转换前:
  count: 42 (str)
  ratio: 3.14 (str)
  enabled: true (str)
  name: test (str)

转换后:
  count: 42 (int)
  ratio: 3.14 (float)
  enabled: True (bool)
  name: test (str)


## 11.6 处理 LLM 的工具调用输出

### 查看 ToolCall 数据模型

In [5]:
from backend.llm.models import ToolCall, FunctionCall
import json

print("=== ToolCall 模型结构 ===\n")
print(json.dumps(ToolCall.model_json_schema(), indent=2, ensure_ascii=False))

=== ToolCall 模型结构 ===

{
  "$defs": {
    "FunctionCall": {
      "description": "函数调用信息",
      "properties": {
        "name": {
          "description": "函数名称",
          "title": "Name",
          "type": "string"
        },
        "arguments": {
          "description": "函数参数（JSON字符串）",
          "title": "Arguments",
          "type": "string"
        }
      },
      "required": [
        "name",
        "arguments"
      ],
      "title": "FunctionCall",
      "type": "object"
    }
  },
  "description": "工具调用模型",
  "properties": {
    "id": {
      "description": "工具调用ID",
      "title": "Id",
      "type": "string"
    },
    "type": {
      "default": "function",
      "description": "调用类型",
      "title": "Type",
      "type": "string"
    },
    "function": {
      "$ref": "#/$defs/FunctionCall"
    }
  },
  "required": [
    "id",
    "function"
  ],
  "title": "ToolCall",
  "type": "object"
}


### 从 LLM 响应中提取工具调用

In [6]:
def extract_tool_calls(response) -> list[ToolCall]:
    """
    从 LLM 响应中提取工具调用
    """
    tool_calls = []
    
    # 检查响应中是否有 tool_calls
    if hasattr(response, 'tool_calls') and response.tool_calls:
        for tool_call in response.tool_calls:
            tool_calls.append(tool_call)
    
    return tool_calls

# 模拟工具调用响应
mock_tool_call = ToolCall(
    id="call_123",
    type="function",
    function=FunctionCall(
        name="calculator",
        arguments='{"expression": "2 + 2"}'
    )
)

print("=== 模拟工具调用 ===\n")
print(f"工具 ID: {mock_tool_call.id}")
print(f"工具名称: {mock_tool_call.function.name}")
print(f"参数 (JSON 字符串): {mock_tool_call.function.arguments}")

# 解析参数
import json
args = json.loads(mock_tool_call.function.arguments)
print(f"\n解析后的参数: {args}")

=== 模拟工具调用 ===

工具 ID: call_123
工具名称: calculator
参数 (JSON 字符串): {"expression": "2 + 2"}

解析后的参数: {'expression': '2 + 2'}


## 11.7 完整的工具调用流程

In [7]:
async def execute_tool_call(tool_call: ToolCall) -> str:
    """
    执行单个工具调用
    """
    # 1. 获取工具
    tool = ToolRegistry.get(tool_call.function.name)
    if not tool:
        return f"错误: 未找到工具 {tool_call.function.name}"
    
    # 2. 解析参数
    try:
        arguments = json.loads(tool_call.function.arguments)
    except json.JSONDecodeError as e:
        return f"错误: 参数解析失败 - {str(e)}"
    
    # 3. 获取 JSON Schema
    schema = tool.get_schema()
    properties = schema.get("properties", {})
    required = schema.get("required", [])
    
    # 4. 验证参数 - 使用 JSON Schema 格式
    is_valid, error = ParameterValidator.validate(arguments, properties, required)
    if not is_valid:
        return f"错误: {error}"
    
    # 5. 类型转换
    arguments = ParameterConverter.convert(arguments, properties)
    
    # 6. 执行工具
    try:
        result = await tool.execute(**arguments)
        return str(result)
    except Exception as e:
        return f"错误: 工具执行失败 - {str(e)}"

# 测试完整流程
async def test_full_flow():
    """测试完整的工具调用流程"""
    
    # 模拟工具调用
    test_calls = [
        ToolCall(
            id="call_1",
            type="function",
            function=FunctionCall(
                name="calculator",
                arguments='{"expression": "123 * 456"}'
            )
        ),
        ToolCall(
            id="call_2",
            type="function",
            function=FunctionCall(
                name="calculator",
                arguments='{}'  # 缺少必填参数
            )
        ),
    ]
    
    print("=== 工具调用执行测试 ===\n")
    
    for call in test_calls:
        print(f"调用: {call.function.name}({call.function.arguments})")
        result = await execute_tool_call(call)
        print(f"结果: {result}\n")

await test_full_flow()

=== 工具调用执行测试 ===

调用: calculator({"expression": "123 * 456"})
结果: 错误: 工具执行失败 - 'ToolResult' object can't be awaited

调用: calculator({})
结果: 错误: 缺少必填参数: expression



## 11.8 高级：智能参数补全

### 场景：LLM 没有提供所有参数

有些参数可以从上下文推断，不需要每次都询问用户。

In [8]:
class SmartParameterCompleter:
    """
    智能参数补全器
    从上下文中推断缺失的参数
    """
    
    def __init__(self, context: Dict[str, Any] = None):
        self.context = context or {}
    
    def complete(self, params: Dict[str, Any], schema: Dict[str, Any]) -> Dict[str, Any]:
        """
        补全缺失的参数
        """
        result = params.copy()
        
        for param_name, param_def in schema.items():
            # 跳过已提供的参数
            if param_name in result:
                continue
            
            # 使用默认值
            if "default" in param_def:
                result[param_name] = param_def["default"]
                continue
            
            # 从上下文中推断
            if param_name in self.context:
                result[param_name] = self.context[param_name]
        
        return result

# 示例：天气查询工具
weather_schema = {
    "city": {"type": "string", "required": True},
    "date": {"type": "string", "required": False, "default": "today"},
    "unit": {"type": "string", "required": False, "enum": ["celsius", "fahrenheit"]}
}

# 设置上下文（用户偏好）
user_context = {
    "unit": "celsius",  # 用户偏好使用摄氏度
    "city": "Beijing"    # 用户默认城市
}

completer = SmartParameterCompleter(user_context)

# 测试补全
test_inputs = [
    {"city": "Shanghai"},  # 只提供城市
    {},                      # 什么都不提供
    {"city": "New York", "unit": "fahrenheit"}  # 全部提供
]

print("=== 智能参数补全测试 ===\n")
for params in test_inputs:
    completed = completer.complete(params, weather_schema)
    print(f"输入: {params}")
    print(f"补全: {completed}")
    print()

=== 智能参数补全测试 ===

输入: {'city': 'Shanghai'}
补全: {'city': 'Shanghai', 'date': 'today', 'unit': 'celsius'}

输入: {}
补全: {'city': 'Beijing', 'date': 'today', 'unit': 'celsius'}

输入: {'city': 'New York', 'unit': 'fahrenheit'}
补全: {'city': 'New York', 'unit': 'fahrenheit', 'date': 'today'}



## 11.9 常见面试问题

**Q: 如何处理 LLM 返回的无效参数？**
- A:
  1. 验证参数：类型、必填、值域
  2. 返回明确的错误信息给 LLM
  3. 让 LLM 重新生成正确的参数
  4. 可以在 system prompt 中强调参数格式

**Q: JSON Schema 在工具调用中的作用是什么？**
- A:
  - 定义参数的结构和类型
  - 传递给 LLM，让它理解如何调用工具
  - 用于验证 LLM 返回的参数
  - 是 Function Calling API 的标准格式

**Q: 为什么需要参数类型转换？**
- A:
  - LLM 输出的都是字符串（JSON 格式）
  - 工具函数可能需要特定的类型（int/float/bool）
  - 转换可以避免类型错误
  - 提升工具调用的成功率

**Q: 如何处理可选参数？**
- A:
  - 在 schema 中标记 `required: false`
  - 在 execute 方法中处理 None 值
  - 设置合理的默认值
  - 使用智能补全从上下文推断

**Q: Tool Call 的 arguments 为什么是字符串？**
- A:
  - 这是 OpenAI Function Calling API 的设计
  - JSON 本身就是字符串格式
  - 需要用 `json.loads()` 解析成字典
  - 这样可以保证传输的兼容性

---

## 总结

本课程学习了参数解析的核心知识：

| 知识点 | 说明 |
|--------|------|
| **JSON Schema** | 标准化的参数定义格式 |
| **参数验证** | 类型、必填、值域、业务逻辑 |
| **类型转换** | 字符串 → 正确的 Python 类型 |
| **ToolCall 解析** | 从 LLM 响应提取工具调用 |
| **智能补全** | 从上下文推断缺失参数 |
| **错误处理** | 明确的错误反馈给 LLM |

**下一步**: L12 将学习如何开发自定义工具！